In [1]:
import os, sys, glob, time, json, subprocess, hashlib, importlib.util

# ====== settings ======
MODE = "test"   # "check" = machinery check on 4 VALIDATION patches (3 models, about 10 min). "test" = FINAL TEST EVALUATION (16 models, 396 test patches).
# ======================
FROZEN_CKPT = {
    "merged_l0.005": "fd66c6ea76b66f1a", "merged_l0.02": "d5a9df203a8ded20", "merged_l0.02_s1": "37175044c91e3b1c",
    "merged_l0.08": "2625ac0069441b2d", "merged_l0.32": "d04ae4c88e5cab00", "merged_l1.28": "11238f565d3dcc1f",
    "split_l0.005": "8cd63293ad7de0bd", "split_l0.01": "14e6cea9c2f9d64a", "split_l0.02": "0fffde7adfa48ee3",
    "split_l0.02_s1": "da18d30a86b4b5d8", "split_l0.04": "a17a5f9137eee139", "split_l0.08": "49c265ec223a8fb6",
    "split_l0.32": "a71ee282e1041c71", "split_l1.28": "1294434b5c5ff6ac",
    "wide_merged_l0.08": "154d285ea8c3e65e", "wide_merged_l1.28": "e1e515ce193b2afb"}
FROZEN_FILES = {"code/dipcoma_eval.py": "e048c6ca53ad6ff1", "code/dipcoma_learned.py": "7234b179095aa731", "protocol.json": "efb752af9201d098"}
CHECK_TAGS = ["merged_l0.02", "split_l0.005", "wide_merged_l0.08"]
# validation results of the same models (bpp, PSNR, SSIM), for the test-against-validation table
VAL = {"merged_l0.005": [0.473, 35.90, 0.892], "merged_l0.02": [0.972, 36.62, 0.904], "merged_l0.02_s1": [0.978, 36.59, 0.910],
       "merged_l0.08": [1.470, 36.79, 0.909], "merged_l0.32": [1.664, 36.31, 0.912], "merged_l1.28": [1.755, 37.28, 0.914],
       "wide_merged_l0.08": [2.030, 37.03, 0.925], "wide_merged_l1.28": [2.747, 37.47, 0.925],
       "split_l0.005": [0.665, 35.26, 0.848], "split_l0.01": [1.057, 35.92, 0.875], "split_l0.02": [1.859, 37.51, 0.906],
       "split_l0.02_s1": [1.842, 37.34, 0.901], "split_l0.04": [2.954, 38.75, 0.925], "split_l0.08": [4.927, 39.30, 0.939],
       "split_l0.32": [10.264, 41.25, 0.961], "split_l1.28": [16.405, 41.52, 0.964]}
SPLIT, N_PATCH = ("val", 4) if MODE == "check" else ("test", 0)
want = CHECK_TAGS if MODE == "check" else list(FROZEN_CKPT)
OUT = "/kaggle/working/final_check" if MODE == "check" else "/kaggle/working/final_test"
os.makedirs(OUT, exist_ok=True)

checks = []
def check(name, ok, info=""):
    checks.append(bool(ok)); print(("PASS" if ok else "FAIL"), "-", name, info, flush=True)
def sha16(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()[:16]
print(f"MODE = {MODE}: split = {SPLIT}", flush=True)

# --- 1. freeze verification: nothing else runs unless this passes ---
roots = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/protocol.json", recursive=True)]
check("exactly one dataset root", len(roots) == 1, str(roots))
found = {}
for p in sorted(glob.glob("/kaggle/input/**/ckpt_*.pt", recursive=True)):
    found.setdefault(os.path.basename(p)[5:-3], p)
if len(roots) == 1:
    ROOT = roots[0]
    bad = []
    for t, h in FROZEN_CKPT.items():
        if t not in found: bad.append((t, "missing"))
        elif sha16(found[t]) != h: bad.append((t, "fingerprint differs"))
    for rel, h in FROZEN_FILES.items():
        if sha16(f"{ROOT}/{rel}") != h: bad.append((rel, "fingerprint differs"))
    check("frozen record verified (16 checkpoints and 3 files unchanged)", not bad, str(bad) if bad else "")

# --- 2. environment ---
if all(checks):
    if importlib.util.find_spec("compressai") is None:
        print("compressai missing, installing...", flush=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "compressai==1.2.8"], capture_output=True, text=True)
        importlib.invalidate_caches()
    check("compressai available", importlib.util.find_spec("compressai") is not None)
    g = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
    ngpu = max(1, min(2, g.stdout.count("GPU ")))
    check("at least 1 GPU visible", g.stdout.count("GPU ") >= 1, f"({g.stdout.count('GPU ')} seen)")
    if MODE == "test":
        n_test = len(glob.glob("/kaggle/input/**/test/*/*.npy", recursive=True))
        check("test dataset attached: 396 patches", n_test == 396, str(n_test))
    else:
        print("INFO - check mode: test patches are NOT read", flush=True)

MAKE_PATHS = r'''
import os, sys, glob, json
split, n, out = sys.argv[1], int(sys.argv[2]), sys.argv[3]
roots = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/protocol.json", recursive=True)]
assert len(roots) == 1, roots
os.environ["DIPCOMA_ROOT"] = roots[0]
sys.path.insert(0, roots[0] + "/code")
import dipcoma_eval as E
paths = E.list_patches(split)
if split == "test":
    idx = {(os.path.basename(os.path.dirname(p)), os.path.basename(p)): p for p in glob.glob("/kaggle/input/**/test/*/*.npy", recursive=True)}
    mapped, missing = [], []
    for p in paths:
        k = (os.path.basename(os.path.dirname(p)), os.path.basename(p))
        (mapped if k in idx else missing).append(idx.get(k, p))
    assert not missing, f"{len(missing)} test patches not found, e.g. {missing[:2]}"
    assert len(idx) == len(paths), (len(idx), len(paths))
    paths = mapped
if n > 0: paths = paths[:n]
json.dump(paths, open(out, "w"))
print(len(paths))
'''
JPEG_LIB = r'''
import io, os, sys, glob, time
import numpy as np
from PIL import Image
roots = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/protocol.json", recursive=True)]
assert len(roots) == 1, roots
os.environ["DIPCOMA_ROOT"] = roots[0]
sys.path.insert(0, roots[0] + "/code")
import dipcoma_eval as E

def jpeg_rows(path, qualities):
    xn = E.normalise(E.load_dn(path))
    q8 = E.to_uint8(xn)
    rows = []
    for q in qualities:
        nbytes, te, td = 0, 0.0, 0.0
        rec = np.empty_like(xn)
        for b in range(E.NB):
            buf = io.BytesIO()
            a = time.time(); Image.fromarray(q8[b]).save(buf, format="JPEG", quality=int(q), optimize=True); te += time.time() - a
            nbytes += buf.tell()
            buf.seek(0)
            a = time.time(); d = np.array(Image.open(buf)); td += time.time() - a
            rec[b] = E.from_uint8(d)
        pb = E.psnr_per_band(xn, rec)
        rows.append(dict(patch=os.path.basename(path), quality=int(q), bytes=nbytes, cr=E.compression_ratio(nbytes),
                         bpp=E.bpp(nbytes), bps=E.bits_per_sample(nbytes), psnr=E.psnr(xn, rec), ssim=E.ssim(xn, rec),
                         t_enc=te, t_dec=td, **{f"psnr_{b}": v for b, v in zip(E.ORDER, pb)}))
    return rows
'''
JPEG_CHECK = r'''
import sys, json, glob
import pandas as pd
from jpeg_lib import jpeg_rows, E
J = pd.read_csv(glob.glob("/kaggle/input/**/results/jpeg_val.csv", recursive=True)[0])
quals = sorted(J.quality.unique())
paths = E.list_patches("val")[:6]
rows = [r for p in paths for r in jpeg_rows(p, quals)]
M = pd.DataFrame(rows).merge(J, on=["patch", "quality"], suffixes=("", "_old"))
print("RESULT " + json.dumps(dict(n=len(M), expected=len(paths) * len(quals),
      rel_bytes=float((abs(M.bytes - M.bytes_old) / M.bytes_old).max()),
      d_psnr=float(abs(M.psnr - M.psnr_old).max()), d_ssim=float(abs(M.ssim - M.ssim_old).max()))))
'''
JPEG_WORKER = r'''
import sys, json, glob
import pandas as pd
from jpeg_lib import jpeg_rows
paths_file, out_csv, part, nparts = sys.argv[1], sys.argv[2], int(sys.argv[3]), int(sys.argv[4])
J = pd.read_csv(glob.glob("/kaggle/input/**/results/jpeg_val.csv", recursive=True)[0])
quals = sorted(J.quality.unique())
paths = json.load(open(paths_file))
rows, done = [], 0
for i, p in enumerate(paths):
    if i % nparts != part: continue
    rows += jpeg_rows(p, quals); done += 1
    if done % 10 == 0: print(f"jpeg part {part}: {done} patches done", flush=True)
pd.DataFrame(rows).to_csv(out_csv, index=False)
print(f"jpeg part {part}: finished", flush=True)
'''
WORKER_CODE = r'''
import os, sys, glob, time, json, gc
gpu, tasks_file, OUT, paths_file = sys.argv[1], sys.argv[2], sys.argv[3], sys.argv[4]
os.environ["CUDA_VISIBLE_DEVICES"] = gpu
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"          # needed for deterministic GPU matrix products
hits = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/protocol.json", recursive=True)]
assert len(hits) == 1, hits
ROOT = hits[0]
os.environ["DIPCOMA_ROOT"] = ROOT
sys.dont_write_bytecode = True
sys.path.insert(0, ROOT + "/code")
import numpy as np, pandas as pd, torch
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)
import dipcoma_eval as E
import dipcoma_learned as L
L.DST = E.DATA
val_p = json.load(open(paths_file))
sync = torch.cuda.synchronize
for t in json.load(open(tasks_file))[gpu]:
    tag, mode = t["tag"], t["mode"]
    if os.path.exists(f"{OUT}/summary_{tag}.json"):
        print(f"{tag}: already done, skipping", flush=True); continue
    t0 = time.time()
    model = L.make_model(13 if mode == "merged" else 1, N=t["N"], M=t["M"]).to("cuda")
    ck = L.load_ckpt(t["ckpt"], model)
    model.eval(); model.update(force=True)
    nparams = sum(p.numel() for p in model.parameters())
    def inputs_of(xn):
        x = torch.from_numpy(xn).to("cuda")
        return [x[None]] if mode == "merged" else [x[b:b + 1][None] for b in range(13)]
    def real(xn):                                          # real compress and decompress
        nbytes, rec, te, td = 0, [], 0.0, 0.0
        for inp in inputs_of(xn):
            sync(); a = time.time(); comp = model.compress(inp); sync(); te += time.time() - a
            nbytes += sum(len(q) for qq in comp["strings"] for q in qq)
            sync(); a = time.time(); d = model.decompress(comp["strings"], comp["shape"]); sync(); td += time.time() - a
            rec.append(d["x_hat"][0])
        return nbytes, torch.cat(rec, dim=0).cpu().numpy(), te, td
    def ideal(xn):                                         # error-free reconstruction, for the consistency check
        rec = [model(inp)["x_hat"][0].clamp(0, 1) for inp in inputs_of(xn)]
        return torch.cat(rec, dim=0).cpu().numpy()
    rows = []
    with torch.no_grad():
        real(E.normalise(E.load_dn(val_p[0])))             # untimed warm-up
        for p in val_p:
            xn = E.normalise(E.load_dn(p))
            nbytes, xr, te, td = real(xn)
            xi = ideal(xn)
            pb = E.psnr_per_band(xn, xr)
            rows.append(dict(patch=os.path.basename(p), bytes=nbytes, psnr=E.psnr(xn, xr), ssim=E.ssim(xn, xr),
                             psnr_ideal=E.psnr(xn, xi), finite=bool(np.isfinite(xr).all()), t_enc=te, t_dec=td,
                             **{b: v for b, v in zip(E.ORDER, pb)}))
    df = pd.DataFrame(rows); df.to_csv(f"{OUT}/final_{tag}.csv", index=False)
    mism = (df.psnr - df.psnr_ideal).abs()
    s = dict(tag=tag, mode=mode, lam=ck["lmbda"], params=nparams, n=len(df),
             bytes_mean=float(df.bytes.mean()), bpp=float(E.bpp(df.bytes.mean())), CR=float(E.compression_ratio(df.bytes.mean())),
             psnr=float(df.psnr.mean()), ssim=float(df.ssim.mean()), psnr_ideal=float(df.psnr_ideal.mean()),
             n_nonfinite=int((~df.finite).sum() + df.psnr.isna().sum()),
             n_mismatch=int(((mism > 0.05) | mism.isna()).sum()),
             max_mismatch=float(np.nanmax(mism)) if mism.notna().any() else float("nan"),
             perband=[float(df[b].mean()) for b in E.ORDER])
    json.dump(s, open(f"{OUT}/summary_{tag}.json", "w"))
    print(f"{tag}: done in {time.time() - t0:.0f} s | bpp {s['bpp']:.3f} | PSNR {s['psnr']:.2f} dB | mismatches {s['n_mismatch']}", flush=True)
    del model; gc.collect(); torch.cuda.empty_cache()
'''
REPORT_CODE = r'''
import os, sys, glob, json
import numpy as np, pandas as pd
OUT, MODE = sys.argv[1], sys.argv[2]
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 60)
ORDER = ["B01", "B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B09", "B10", "B11", "B12"]
checks = []
def check(name, ok, info=""):
    checks.append(bool(ok)); print(("PASS" if ok else "FAIL"), "-", name, info)
tasks = json.load(open(f"{OUT}/tasks.json"))
tags = [t["tag"] for g in tasks.values() for t in g]
n_expected = len(json.load(open(f"{OUT}/paths.json")))
VAL = json.load(open(f"{OUT}/val_ref.json"))
S = [json.load(open(f"{OUT}/summary_{t}.json")) for t in tags if os.path.exists(f"{OUT}/summary_{t}.json")]
check("all models evaluated", len(S) == len(tags), f"{len(S)}/{len(tags)}")
fam = lambda s: 0 if s["tag"].startswith("merged") else (1 if s["tag"].startswith("wide") else 2)
S.sort(key=lambda s: (fam(s), s["lam"], s["tag"].endswith("_s1")))
check("every model decoded every patch", all(s["n"] == n_expected for s in S), f"({n_expected} patches)")
check("no decode produced non-finite values", sum(s["n_nonfinite"] for s in S) == 0, f"({sum(s['n_nonfinite'] for s in S)} patches)")
check("every decoded patch matches the error-free PSNR within 0.05 dB", sum(s["n_mismatch"] for s in S) == 0,
      f"({sum(s['n_mismatch'] for s in S)} patches differ; largest {max(s['max_mismatch'] for s in S):.3f} dB)")

parts = sorted(glob.glob(f"{OUT}/jpeg_part*.csv"))
J = pd.concat([pd.read_csv(p) for p in parts]) if parts else pd.DataFrame()
okj = len(J) > 0 and J.patch.nunique() == n_expected and J.groupby("quality").patch.nunique().min() == n_expected
check("JPEG results complete for every patch and quality", okj)
pbc = [f"psnr_{b}" for b in ORDER]
Jq = J.groupby("quality").agg(bpp=("bpp", "mean"), psnr=("psnr", "mean"), ssim=("ssim", "mean"), **{c: (c, "mean") for c in pbc}).reset_index().sort_values("bpp")
Jq["val_psnr"] = Jq.quality.map({1: 27.43, 10: 33.98, 20: 36.68, 30: 38.09, 50: 39.76, 70: 41.23, 85: 43.51, 95: 47.86})
print("\nJPEG on these patches (mean over patches; val_psnr = value on the validation patches):")
print(Jq[["quality", "bpp", "psnr", "ssim", "val_psnr"]].round(3).to_string(index=False))
Ju = Jq.drop_duplicates(subset="bpp")
def jint(b, col):
    if b < Ju.bpp.min() or b > Ju.bpp.max(): return np.nan
    return float(np.interp(np.log(b), np.log(Ju.bpp.values), Ju[col].values))

t1 = pd.DataFrame([dict(model=s["tag"], params_M=s["params"] / 1e6, bpp=s["bpp"], CR=s["CR"], PSNR=s["psnr"], SSIM=s["ssim"]) for s in S])
print(f"\nTable 1: results on the {MODE.upper()} patches, real deterministic decoding:")
print(t1.round({"params_M": 2, "bpp": 3, "CR": 0, "PSNR": 2, "SSIM": 3}).to_string(index=False))
t2 = pd.DataFrame([dict(model=s["tag"], bpp=s["bpp"], PSNR=s["psnr"], JPEG_PSNR=jint(s["bpp"], "psnr"), SSIM=s["ssim"], JPEG_SSIM=jint(s["bpp"], "ssim")) for s in S])
t2["dPSNR"], t2["dSSIM"] = t2.PSNR - t2.JPEG_PSNR, t2.SSIM - t2.JPEG_SSIM
print("\nTable 2: against JPEG at the same bpp (JPEG interpolated in log bpp; nan = outside JPEG's range):")
print(t2.round({"bpp": 3, "PSNR": 2, "JPEG_PSNR": 2, "dPSNR": 2, "SSIM": 3, "JPEG_SSIM": 3, "dSSIM": 3}).to_string(index=False))
t3 = pd.DataFrame([[s["tag"]] + s["perband"] for s in S], columns=["model"] + ORDER)
print("\nTable 3: mean PSNR (dB) per band:")
print(t3.round(1).to_string(index=False))
t4 = pd.DataFrame([[s["tag"]] + [s["perband"][i] - jint(s["bpp"], f"psnr_{b}") for i, b in enumerate(ORDER)] for s in S], columns=["model"] + ORDER)
print("\nTable 4: per-band PSNR minus JPEG per-band PSNR at the same bpp (dB, positive = learned codec better):")
print(t4.round(1).to_string(index=False))
G = {"10m": ["B02", "B03", "B04", "B08"], "20m": ["B05", "B06", "B07", "B8A", "B11", "B12"], "60m": ["B01", "B09", "B10"]}
t5 = pd.DataFrame({"model": t4["model"], **{f"{k} bands": t4[v].mean(axis=1) for k, v in G.items()}})
print("\nTable 5: the same difference, averaged by native band resolution:")
print(t5.round(1).to_string(index=False))
if MODE == "test":
    t6 = pd.DataFrame([dict(model=s["tag"], val_bpp=VAL[s["tag"]][0], test_bpp=s["bpp"], val_PSNR=VAL[s["tag"]][1], test_PSNR=s["psnr"],
                            dPSNR=s["psnr"] - VAL[s["tag"]][1], val_SSIM=VAL[s["tag"]][2], test_SSIM=s["ssim"]) for s in S])
    print("\nTable 6: test against validation (same frozen models):")
    print(t6.round({"val_bpp": 3, "test_bpp": 3, "val_PSNR": 2, "test_PSNR": 2, "dPSNR": 2, "val_SSIM": 3, "test_SSIM": 3}).to_string(index=False))
    t6.to_csv(f"{OUT}/table6.csv", index=False)
for n, t in (("table1", t1), ("table2", t2), ("table3", t3), ("table4", t4), ("table5", t5)):
    t.to_csv(f"{OUT}/{n}.csv", index=False)
Jq.to_csv(f"{OUT}/jpeg_summary.csv", index=False)
print("\nREPORT DONE")
'''
if all(checks):
    for name, code in (("make_paths.py", MAKE_PATHS), ("jpeg_lib.py", JPEG_LIB), ("jpeg_check.py", JPEG_CHECK),
                       ("jpeg_worker.py", JPEG_WORKER), ("worker.py", WORKER_CODE), ("report.py", REPORT_CODE)):
        open(f"{OUT}/{name}", "w").write(code)
    json.dump(VAL, open(f"{OUT}/val_ref.json", "w"))
    r = subprocess.run([sys.executable, f"{OUT}/make_paths.py", SPLIT, str(N_PATCH), f"{OUT}/paths.json"], capture_output=True, text=True)
    check("patch list built", r.returncode == 0, r.stdout.strip() + " patches" if r.returncode == 0 else r.stderr[-500:])

if all(checks):
    def cost(tag): return 700 if tag.startswith("split") else (100 if tag.startswith("wide") else 50)   # rough seconds, only for balancing
    tasks, load = {str(i): [] for i in range(ngpu)}, [0] * ngpu
    for tag in sorted(want, key=cost, reverse=True):
        i = load.index(min(load)); wide = tag.startswith("wide")
        tasks[str(i)].append(dict(tag=tag, ckpt=found[tag], mode="split" if tag.startswith("split") else "merged",
                                  N=192 if wide else 128, M=320 if wide else 192))
        load[i] += cost(tag)
    json.dump(tasks, open(f"{OUT}/tasks.json", "w"))
    r = subprocess.run([sys.executable, f"{OUT}/jpeg_check.py", OUT], capture_output=True, text=True, cwd=OUT)
    line = [l for l in r.stdout.splitlines() if l.startswith("RESULT ")]
    res = json.loads(line[-1][7:]) if line else {}
    print("JPEG reproduction details:", res if res else r.stderr[-500:], flush=True)
    check("JPEG code reproduces the validation JPEG results (bytes within 0.5%, PSNR within 0.05 dB)",
          bool(res) and res["n"] == res["expected"] and res["rel_bytes"] < 0.005 and res["d_psnr"] < 0.05)

if all(checks):
    T0 = time.time()
    procs = {}
    for gi in tasks:
        log = open(f"{OUT}/log_gpu{gi}.txt", "a")
        procs[f"gpu{gi}"] = subprocess.Popen([sys.executable, f"{OUT}/worker.py", gi, f"{OUT}/tasks.json", OUT, f"{OUT}/paths.json"],
                                             stdout=log, stderr=subprocess.STDOUT, env=dict(os.environ, PYTHONUNBUFFERED="1"))
        print(f"started GPU {gi}: {[t['tag'] for t in tasks[gi]]}", flush=True)
    for part in range(2):
        log = open(f"{OUT}/log_jpeg{part}.txt", "a")
        procs[f"jpeg{part}"] = subprocess.Popen([sys.executable, f"{OUT}/jpeg_worker.py", f"{OUT}/paths.json", f"{OUT}/jpeg_part{part}.csv", str(part), "2"],
                                                stdout=log, stderr=subprocess.STDOUT, cwd=OUT, env=dict(os.environ, PYTHONUNBUFFERED="1"))
    def last(name, key):
        ls = [l for l in open(f"{OUT}/log_{name}.txt").read().strip().splitlines() if key in l or "Error" in l]
        return ls[-1] if ls else "nothing yet"
    lastp = 0.0
    while any(p.poll() is None for p in procs.values()):
        time.sleep(10)
        if time.time() - lastp > 300:
            lastp = time.time()
            print(f"--- {(time.time() - T0) / 60:.1f} min elapsed ---", flush=True)
            for n in procs:
                print(f"  {n}:", last(n, "done in" if n.startswith("gpu") else "jpeg part"), flush=True)
    ok_exit = all(p.returncode == 0 for p in procs.values())
    check("all workers exited with code 0", ok_exit, str({n: p.returncode for n, p in procs.items()}))
    if not ok_exit:
        for n in procs: print(f"--- log {n} (last 12 lines) ---"); print(*open(f"{OUT}/log_{n}.txt").read().splitlines()[-12:], sep="\n")
    r = subprocess.run([sys.executable, f"{OUT}/report.py", OUT, MODE], capture_output=True, text=True)
    print(r.stdout)
    if r.returncode: print(r.stderr[-1500:])
    check("report finished", r.returncode == 0 and "REPORT DONE" in r.stdout)
    check("report has no FAIL line", "FAIL" not in r.stdout)
    print(f"INFO - wall-clock {(time.time() - T0) / 60:.1f} min")

print("\nALL PASS" if all(checks) else "\nSOME CHECKS FAILED - paste the whole output here")

MODE = test: split = test
PASS - exactly one dataset root ['/kaggle/input/datasets/krrishshandilya/dipcoma-trainval']
PASS - frozen record verified (16 checkpoints and 3 files unchanged) 
compressai missing, installing...
PASS - compressai available 
PASS - at least 1 GPU visible (2 seen)
PASS - test dataset attached: 396 patches 396
PASS - patch list built 396 patches
JPEG reproduction details: {'n': 78, 'expected': 78, 'rel_bytes': 0.0, 'd_psnr': 2.1316282072803006e-14, 'd_ssim': 1.1102230246251565e-16}
PASS - JPEG code reproduces the validation JPEG results (bytes within 0.5%, PSNR within 0.05 dB) 
started GPU 0: ['split_l0.005', 'split_l0.02', 'split_l0.04', 'split_l0.32', 'wide_merged_l0.08', 'merged_l0.005', 'merged_l0.02_s1', 'merged_l0.32']
started GPU 1: ['split_l0.01', 'split_l0.02_s1', 'split_l0.08', 'split_l1.28', 'wide_merged_l1.28', 'merged_l0.02', 'merged_l0.08', 'merged_l1.28']
--- 0.2 min elapsed ---
  gpu0: nothing yet
  gpu1: nothing yet
  jpeg0: nothing yet
  jpeg1: